In [1]:
import pandas as pd
from jobspy import scrape_jobs

search_terms = [
    "AI Engineer",
    "LLM Engineer",
    "Data Scientist",
    "Machine Learning Engineer"
]

all_jobs = []


In [ ]:
for term in search_terms:
    print(f"Scraping: {term}")

    jobs = scrape_jobs(
        site_name=[ "linkedin"],
        search_term=term,
        location="India",
        results_wanted=50
    )

    if not jobs.empty:
        all_jobs.append(jobs)

if all_jobs:
    df = pd.concat(all_jobs, ignore_index=True)
    columns_to_keep = [
        "site",
        "title",
        "company",
        "location",
        "date_posted",
        "min_amount",
        "max_amount",
        "interval"
    ]

    filtered_df = df[
        [col for col in columns_to_keep if col in df.columns]
    ]

    filtered_df.to_csv("linkedin.csv", index=False)

    print(filtered_df.head())
    print(f"\nTotal Jobs Scraped: {len(filtered_df)}")
else:
    print("No jobs found.")


Scraping: AI Engineer


2026-06-24 16:56:09,654 - INFO - JobSpy:Linkedin - finished scraping


Scraping: LLM Engineer
Scraping: Data Scientist
Scraping: Machine Learning Engineer
       site                                  title       company  \
0  linkedin                         AI/ML Engineer         Chubb   
1  linkedin                     Python ML Engineer       Infosys   
2  linkedin               Software Engineer, AI/ML        Google   
3  linkedin                         AI/ML Engineer       Infosys   
4  linkedin  Software Engineer - AI/ML with Python  HARMAN India   

                           location date_posted min_amount max_amount interval  
0       Bengaluru, Karnataka, India  2026-06-23       None       None     None  
1  Bengaluru East, Karnataka, India  2026-06-22       None       None     None  
2       Bengaluru, Karnataka, India  2026-06-20       None       None     None  
3  Bengaluru East, Karnataka, India  2026-06-20       None       None     None  
4       Bengaluru, Karnataka, India  2026-06-19       None       None     None  

Total Jobs Scraped: 

In [ ]:
import pandas as pd
import numpy as np
from jobspy import scrape_jobs


search_terms = [
    "AI Engineer",
    "LLM Engineer",
    "Data Scientist",
    "Machine Learning Engineer"
]

all_jobs = []


for term in search_terms:
    print(f"Scraping: {term}")
    
    try:
        jobs = scrape_jobs(
            site_name=["linkedin"],
            search_term=term,
            location="India",
            results_wanted=50
        )
        
        if not jobs.empty:
            
            jobs['searched_role'] = term
            all_jobs.append(jobs)
            
    except Exception as e:
        print(f"Error scraping {term}: {e}")


if all_jobs:
    df = pd.concat(all_jobs, ignore_index=True)
    
    
    columns_to_keep = [
        "site",
        "title",
        "company",
        "location",
        "date_posted",
        #"min_amount",
        #"max_amount",
        #"interval",
        #"currency",
        "searched_role"
    ]
    
    
    filtered_df = df[[col for col in columns_to_keep if col in df.columns]].copy()


    def estimate_india_salary(row):
        
        if 'min_amount' in row and pd.notna(row['min_amount']) and row['min_amount'] > 0:
            interval = f" {row['interval']}" if pd.notna(row['interval']) else ""
            currency = row.get('currency', '₹')
            return f"{currency}{row['min_amount']} - {row.get('max_amount', '')}{interval}"
            
        title = str(row.get('title', '')).lower()
        role = str(row.get('searched_role', '')).lower()
        
        
        salary_bands = {
            "ai engineer": (8, 18, 35),
            "llm engineer": (12, 24, 42),
            "machine learning engineer": (9, 20, 38),
            "data scientist": (7, 16, 32)
        }
        
       
        base, senior, lead = salary_bands.get(role, (8, 16, 30))
        
        
        if any(word in title for word in ['intern', 'associate', 'analyst', '1', 'i ']):
            min_lpa, max_lpa = max(3, int(base * 0.6)), int(base * 1.0)
        elif any(word in title for word in ['principal', 'manager', 'director', 'gm', 'lead', 'architect', 'vp', 'avp']):
            min_lpa, max_lpa = int(lead * 0.8), int(lead * 1.4)
        elif any(word in title for word in ['senior', 'sr', 'ii', 'staff', 'expert']):
            min_lpa, max_lpa = int(senior * 0.85), int(senior * 1.3)
        else:
            
            min_lpa, max_lpa = int(base * 0.9), int(senior * 1.1)
    
        return f"₹{min_lpa}L - ₹{max_lpa}L PA"

    
    filtered_df['salary'] = filtered_df.apply(estimate_india_salary, axis=1)
    
    
    output_filename = "linkedin_with_salaries.csv"
    filtered_df.to_csv(output_filename, index=False)
    
    print("\n" + "="*50)
    print("SUCCESS: Populated missing salaries using market tier estimates!")
    print("="*50 + "\n")
    
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    
    preview_cols = ["title", "company", "location", "searched_role", "salary"]
    print(filtered_df[preview_cols].head(10)) 
    
    print(f"\nTotal Jobs Scraped & Processed: {len(filtered_df)}")
else:
    print("No jobs found during scraping execution.")

Scraping: AI Engineer
Scraping: LLM Engineer
Scraping: Data Scientist
Scraping: Machine Learning Engineer

SUCCESS: Populated missing salaries using market tier estimates!

                                    title                 company                           location searched_role         salary
0                Python + AI/ML Developer              Terralogic                                      AI Engineer  ₹7L - ₹19L PA
1                     AI Developer Intern                UPTITUDE           Gurugram, Haryana, India   AI Engineer   ₹4L - ₹8L PA
2                            AI Developer  FUJITSU LIMITED（JAPAN）  Bangalore Urban, Karnataka, India   AI Engineer   ₹4L - ₹8L PA
3                          AI/ML Engineer                   Chubb        Bengaluru, Karnataka, India   AI Engineer  ₹7L - ₹19L PA
4  Associate Engineer - AI/ML with Python            HARMAN India        Bengaluru, Karnataka, India   AI Engineer   ₹4L - ₹8L PA
5         Python AI specialist programmer      